In [ ]:
# Example 1: Basic Serialization and Advanced Object Types
import dill
import pickle
import types

# Test basic serialization vs pickle
print("Comparing dill vs standard pickle for complex objects:")
print("-" * 50)

# Create objects that pickle cannot handle
def outer_function(x):
    """Function with closure that pickle cannot serialize"""
    def inner_function(y):
        return x + y + 10
    return inner_function

# Create a closure
closure_func = outer_function(5)

# Lambda function
lambda_func = lambda x: x * 2 + 1

# Generator function
def generator_func():
    for i in range(3):
        yield i ** 2

# Test objects, each with the exception pickle and dill are EXPECTED to raise (None = it must serialize).
# pickle stores functions by reference, so a closure or a lambda cannot be found again by name; a
# module-level function pickles fine. A live generator's frame cannot be serialized by either library.
# pickle raises PicklingError for both on Python 3.14; on earlier versions the closure raises
# AttributeError ("Can't pickle local object") and the lambda PicklingError.
test_objects = [
    ("closure function", closure_func, (pickle.PicklingError, AttributeError), None),
    ("lambda function", lambda_func, (pickle.PicklingError, AttributeError), None),
    ("generator function", generator_func, None, None),
    ("generator object", generator_func(), TypeError, TypeError),
]

for obj_name, obj, pickle_expected, dill_expected in test_objects:
    print(f"\nTesting {obj_name}:")
    
    # Test with standard pickle
    try:
        pickle_data = pickle.dumps(obj)
        pickle_restored = pickle.loads(pickle_data)
    except (pickle.PicklingError, TypeError, AttributeError) as e:
        if pickle_expected is None or not isinstance(e, pickle_expected):
            raise
        print(f"  ✗ Pickle: Failed as expected - {type(e).__name__}")
    else:
        assert pickle_expected is None, f"pickle unexpectedly serialized the {obj_name}"
        print(f"  ✓ Pickle: Success")
    
    # Test with dill
    try:
        dill_data = dill.dumps(obj)
        dill_restored = dill.loads(dill_data)
    except (pickle.PicklingError, TypeError, AttributeError) as e:
        if dill_expected is None or not isinstance(e, dill_expected):
            raise
        print(f"  ✗ Dill: Failed as expected - {type(e).__name__}")
    else:
        assert dill_expected is None, f"dill unexpectedly serialized the {obj_name}"
        print(f"  ✓ Dill: Success (size: {len(dill_data)} bytes)")
        
        # Test functionality if callable
        if obj_name == "closure function":
            result = dill_restored(3)  # Should return 5 + 3 + 10 = 18
            print(f"    Restored function result: {result}")
            assert result == 18, result
        elif obj_name == "lambda function":
            result = dill_restored(5)  # Should return 5 * 2 + 1 = 11
            print(f"    Restored lambda result: {result}")
            assert result == 11, result
        elif obj_name == "generator function":
            assert list(dill_restored()) == [0, 1, 4]

print("\nBasic serialization comparison completed successfully!")
print("Dill can serialize objects that standard pickle cannot handle!")

In [ ]:
# Example 2: Class and Method Serialization
import dill
import types
from datetime import datetime

# Define complex classes with various method types
class DataProcessor:
    """Complex class with various method types"""
    
    class_variable = "shared_data"
    
    def __init__(self, name, multiplier=2):
        self.name = name
        self.multiplier = multiplier
        self.created_at = datetime.now()
        self._private_data = []
    
    def process_data(self, data):
        """Instance method"""
        return [x * self.multiplier for x in data]
    
    @classmethod
    def create_default(cls):
        """Class method"""
        return cls("default_processor", 3)
    
    @staticmethod
    def validate_data(data):
        """Static method"""
        return all(isinstance(x, (int, float)) for x in data)
    
    @property
    def info(self):
        """Property method"""
        return f"{self.name} (multiplier: {self.multiplier})"
    
    def __call__(self, data):
        """Make instance callable"""
        return self.process_data(data)

# Create instance with state
processor = DataProcessor("test_processor", 5)
processor._private_data = [1, 2, 3, 4]

print("Class and method serialization demonstration:")
print("=" * 45)

# Test serialization of different components
test_items = [
    ("Complete class definition", DataProcessor),
    ("Class instance with state", processor),
    ("Instance method (bound)", processor.process_data),
    ("Class method", DataProcessor.create_default),
    ("Static method", DataProcessor.validate_data),
    ("Property getter", DataProcessor.info.fget),
]

serialized_items = {}

for item_name, item in test_items:
    # Serialize with dill
    serialized = dill.dumps(item)
    deserialized = dill.loads(serialized)
    serialized_items[item_name] = deserialized
    
    print(f"✓ {item_name}: Success ({len(serialized)} bytes)")
    

# every item above is dill-serializable (all 6 round-trip), so a failed round trip fails the cell
assert list(serialized_items) == [name for name, _ in test_items], list(serialized_items)
assert serialized_items["Static method"]([1, 2.5]) is True
assert serialized_items["Property getter"](processor) == "test_processor (multiplier: 5)"

# Test functionality of deserialized objects
print(f"\nTesting deserialized object functionality:")

if "Complete class definition" in serialized_items:
    RestoredClass = serialized_items["Complete class definition"]
    new_instance = RestoredClass("restored", 10)
    result = new_instance.process_data([1, 2, 3])
    print(f"  Restored class works: {result}")
    assert result == [10, 20, 30], result

if "Class instance with state" in serialized_items:
    restored_instance = serialized_items["Class instance with state"]
    print(f"  Restored instance info: {restored_instance.info}")
    print(f"  Restored private data: {restored_instance._private_data}")
    assert restored_instance.info == "test_processor (multiplier: 5)", restored_instance.info
    assert restored_instance._private_data == [1, 2, 3, 4], restored_instance._private_data
    result = restored_instance([2, 4, 6])
    print(f"  Callable instance result: {result}")
    assert result == [10, 20, 30], result

if "Instance method (bound)" in serialized_items:
    restored_method = serialized_items["Instance method (bound)"]
    result = restored_method([3, 6, 9])
    print(f"  Restored bound method result: {result}")
    assert result == [15, 30, 45], result

if "Class method" in serialized_items:
    restored_classmethod = serialized_items["Class method"]
    new_obj = restored_classmethod()
    print(f"  Restored class method created: {new_obj.info}")
    assert new_obj.info == "default_processor (multiplier: 3)", new_obj.info

# Demonstrate complex inheritance
class AdvancedProcessor(DataProcessor):
    def __init__(self, name, multiplier=2, offset=0):
        super().__init__(name, multiplier)
        self.offset = offset
    
    def process_data(self, data):
        """Override parent method"""
        base_result = super().process_data(data)
        return [x + self.offset for x in base_result]

advanced = AdvancedProcessor("advanced", 3, 10)
advanced_serialized = dill.dumps(advanced)
advanced_restored = dill.loads(advanced_serialized)

result = advanced_restored.process_data([1, 2, 3])
print(f"  Inherited class result: {result}")
assert result == [13, 16, 19], result

print("\nClass and method serialization completed successfully!")

In [ ]:
# Example 3: File Operations and Persistent Storage
import dill
import tempfile
import os
from pathlib import Path

# Create complex objects to store
class ConfigManager:
    def __init__(self, app_name):
        self.app_name = app_name
        self.settings = {}
        self.callbacks = []
    
    def add_setting(self, key, value):
        self.settings[key] = value
    
    def add_callback(self, callback):
        self.callbacks.append(callback)
    
    def process_settings(self):
        for callback in self.callbacks:
            for key, value in self.settings.items():
                callback(key, value)

# Create instances with complex state
config = ConfigManager("MyApp")
config.add_setting("debug", True)
config.add_setting("max_users", 1000)

# Add lambda callbacks (standard pickle cannot handle these)
config.add_callback(lambda k, v: print(f"Setting {k} = {v}"))
config.add_callback(lambda k, v: print(f"Validating {k}: {v}") if k == "debug" else None)

# Complex nested data with functions
complex_data = {
    'config': config,
    'processors': [
        lambda x: x * 2,
        lambda x: x + 10,
        lambda x: x ** 0.5
    ],
    'validator': lambda data: all(isinstance(x, (int, float)) for x in data),
    'metadata': {
        'created_by': lambda: "dill_example",
        'version': 1.0
    }
}

print("File operations and persistent storage:")
print("=" * 42)

# Create temporary files for testing
with tempfile.TemporaryDirectory() as temp_dir:
    temp_path = Path(temp_dir)
    
    # Test 1: Save to file using dill.dump
    dill_file = temp_path / "complex_data.dill"
    
    print(f"1. Saving complex data to file:")
    with open(dill_file, 'wb') as f:
        dill.dump(complex_data, f)
    
    file_size = dill_file.stat().st_size
    print(f"   ✓ Saved to {dill_file.name} ({file_size} bytes)")
    
    # Load back from file
    with open(dill_file, 'rb') as f:
        loaded_data = dill.load(f)
    
    print(f"   ✓ Successfully loaded from file")
    
    # Test functionality of loaded objects
    loaded_config = loaded_data['config']
    print(f"   ✓ Config app name: {loaded_config.app_name}")
    print(f"   ✓ Settings count: {len(loaded_config.settings)}")
    
    # Test lambda processors
    test_value = 5
    results = [proc(test_value) for proc in loaded_data['processors']]
    print(f"   ✓ Processors work: {test_value} -> {results}")
    
    # Test validator
    is_valid = loaded_data['validator']([1, 2, 3.5])
    print(f"   ✓ Validator works: {is_valid}")
    assert (loaded_config.app_name, len(loaded_config.settings)) == ("MyApp", 2), loaded_config.settings
    assert results == [10, 15, 5 ** 0.5], results
    assert is_valid is True
    
    
    # Test 2: Multiple objects in separate files
    print(f"\n2. Saving multiple objects separately:")
    
    objects_to_save = {
        'config_only.dill': config,
        'processors_only.dill': complex_data['processors'],
        'metadata_only.dill': complex_data['metadata']
    }
    
    saved_files = {}
    for filename, obj in objects_to_save.items():
        file_path = temp_path / filename
        with open(file_path, 'wb') as f:
            dill.dump(obj, f)
        
        size = file_path.stat().st_size
        saved_files[filename] = size
        print(f"   ✓ Saved {filename} ({size} bytes)")
        
    
    assert saved_files.keys() == objects_to_save.keys(), saved_files

    # Test 3: Load and combine objects
    print(f"\n3. Loading and combining objects:")
    
    # Load config
    with open(temp_path / 'config_only.dill', 'rb') as f:
        loaded_config = dill.load(f)
    
    # Load processors  
    with open(temp_path / 'processors_only.dill', 'rb') as f:
        loaded_processors = dill.load(f)
    
    # Test combined functionality
    loaded_config.process_settings()  # This will use the lambda callbacks
    
    combined_result = sum(proc(10) for proc in loaded_processors)
    print(f"   ✓ Combined processing result: {combined_result}")
    assert combined_result == 20 + 20 + 10 ** 0.5, combined_result
    
    
    # Test 4: Dill settings and compression
    print(f"\n4. Testing dill settings and features:")
    
    # Test with different protocols
    for protocol in [dill.HIGHEST_PROTOCOL, dill.DEFAULT_PROTOCOL]:
        data = dill.dumps(complex_data, protocol=protocol)
        restored = dill.loads(data)
        print(f"   ✓ Protocol {protocol}: {len(data)} bytes")
        assert restored.keys() == complex_data.keys() and restored["processors"][1](5) == 15, restored.keys()
        
    
    # Show dill settings
    print(f"   Dill version: {dill.__version__}")
    print(f"   Highest protocol: {dill.HIGHEST_PROTOCOL}")
    print(f"   Can handle functions: {hasattr(dill, 'loads')}")

print("\nFile operations and persistent storage completed successfully!")

In [ ]:
# Example 4: Distributed Computing and Multiprocessing Integration
import dill
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import threading
import time

# Define functions and classes for distributed processing
def complex_function_with_closure(multiplier):
    """Create a function with closure for distributed processing"""
    def inner_function(data_list):
        # Complex processing with closure variable
        result = []
        for item in data_list:
            if isinstance(item, (int, float)):
                processed = item * multiplier + sum(range(1, int(abs(item)) + 1))
                result.append(processed)
            else:
                result.append(str(item).upper())
        return result
    return inner_function

class DistributedProcessor:
    """Class for distributed processing that needs serialization"""
    
    def __init__(self, name, config):
        self.name = name
        self.config = config
        self.process_count = 0
    
    def process_batch(self, data_batch):
        """Process a batch of data"""
        self.process_count += 1
        
        # Apply configuration-based processing
        multiplier = self.config.get('multiplier', 1)
        offset = self.config.get('offset', 0)
        
        processed = []
        for item in data_batch:
            if isinstance(item, (int, float)):
                result = (item * multiplier) + offset
                processed.append(result)
            else:
                processed.append(f"STR_{item}")
        
        return {
            'processor': self.name,
            'batch_size': len(data_batch),
            'results': processed,
            'process_count': self.process_count
        }
    
    def __call__(self, data):
        """Make instance callable for easy multiprocessing"""
        return self.process_batch(data)

print("Distributed computing and multiprocessing integration:")
print("=" * 54)

# Test 1: Multiprocessing with functions that have closures
print("1. Multiprocessing with closure functions:")

# Create processors with different closures
processors = [
    complex_function_with_closure(2),    # multiplier = 2
    complex_function_with_closure(5),    # multiplier = 5
    complex_function_with_closure(10)    # multiplier = 10
]

# Test data
test_data = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, "text", 12]
]

# Use ProcessPoolExecutor with dill serialization
# Note: This would normally require setting the serialization method
# For demonstration, we'll show how dill enables this

print("   Serializing closure functions with dill:")
for i, processor in enumerate(processors):
    serialized = dill.dumps(processor)
    restored = dill.loads(serialized)
    
    # Test restored function
    result = restored(test_data[i])
    print(f"     Processor {i+1}: {result}")
    assert result == [[3, 7, 12, 18], [40, 51, 63, 76], [135, 155, "TEXT", 198]][i], result


# Test 2: Class-based distributed processing
print(f"\n2. Distributed processing with serializable classes:")

# Create processor instances
processors = [
    DistributedProcessor("proc_1", {'multiplier': 2, 'offset': 10}),
    DistributedProcessor("proc_2", {'multiplier': 3, 'offset': 5}),
    DistributedProcessor("proc_3", {'multiplier': 1, 'offset': 0})
]

# Test serialization of class instances
batches = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9, "test"]
]

print("   Testing class instance serialization:")
for i, (processor, batch) in enumerate(zip(processors, batches)):
    # Serialize processor
    serialized = dill.dumps(processor)
    restored_processor = dill.loads(serialized)
    
    # Process batch with restored processor
    result = restored_processor(batch)
    print(f"     {result['processor']}: Processed {result['batch_size']} items -> {result['results']}")
    assert result["results"] == [[12, 14, 16], [17, 20, 23], [7, 8, 9, "STR_test"]][i], result
    

# Test 3: Thread-safe serialization
print(f"\n3. Thread-safe serialization:")

def thread_serialization_test(thread_id, shared_results):
    """Function to test serialization in different threads"""
    
    # Create unique processor for this thread
    processor = DistributedProcessor(f"thread_{thread_id}", 
                                   {'multiplier': thread_id, 'offset': thread_id * 5})
    
    try:
        # Serialize and deserialize
        start_time = time.time()
        serialized = dill.dumps(processor)
        restored = dill.loads(serialized)
        
        # Test functionality
        test_data = [thread_id, thread_id * 2, thread_id * 3]
        result = restored(test_data)
        
        elapsed = time.time() - start_time
        
        shared_results[thread_id] = {
            'success': True,
            'elapsed': elapsed,
            'data_size': len(serialized),
            'result': result['results']
        }
        
    except Exception as e:
        shared_results[thread_id] = {
            'success': False,
            'error': str(e)
        }

# Run concurrent serialization tests
shared_results = {}
threads = []

for thread_id in range(1, 4):
    thread = threading.Thread(target=thread_serialization_test, 
                            args=(thread_id, shared_results))
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

# Display results: every thread must have written a result, and every round-trip must have succeeded
assert sorted(shared_results) == [1, 2, 3], f"threads that reported: {sorted(shared_results)}"
for thread_id, result in shared_results.items():
    if result['success']:
        print(f"   Thread {thread_id}: Success in {result['elapsed']:.4f}s "
              f"({result['data_size']} bytes) -> {result['result']}")
        # the restored processor has multiplier t and offset 5t, applied to [t, 2t, 3t]
        assert result['result'] == [x * thread_id + thread_id * 5 for x in (thread_id, thread_id * 2, thread_id * 3)], result
    else:
        raise AssertionError(f"Thread {thread_id}: dill round-trip failed - {result['error']}")

# Test 4: Complex nested serialization
print(f"\n4. Complex nested object serialization:")

# Create complex nested structure
complex_workflow = {
    'name': 'distributed_ml_pipeline',
    'stages': [
        {
            'name': 'preprocessing',
            'function': lambda data: [x * 2 for x in data if isinstance(x, (int, float))],
            'config': {'normalize': True}
        },
        {
            'name': 'feature_extraction', 
            'function': lambda data: [x + 10 for x in data],
            'config': {'feature_count': 5}
        },
        {
            'name': 'prediction',
            'function': lambda data: sum(data) / len(data) if data else 0,
            'config': {'model_type': 'linear'}
        }
    ],
    'processor': DistributedProcessor("workflow_proc", {'multiplier': 1, 'offset': 0})
}

# Serialize entire workflow
workflow_serialized = dill.dumps(complex_workflow)
workflow_restored = dill.loads(workflow_serialized)

print(f"   ✓ Complex workflow serialized: {len(workflow_serialized)} bytes")

# Test pipeline execution
test_input = [1, 2, 3, 4, 5]
pipeline_result = test_input

for stage in workflow_restored['stages']:
    stage_func = stage['function']
    pipeline_result = stage_func(pipeline_result)
    print(f"     Stage '{stage['name']}': {pipeline_result}")

assert pipeline_result == 16.0, pipeline_result
print(f"   ✓ Pipeline executed successfully")


print("\nDistributed computing integration completed successfully!")

In [ ]:
# Example 5: Production Patterns and Advanced Features
import dill
import sys
import types
import inspect
from functools import wraps
import time

# Advanced feature: Custom serialization hooks
class SerializationManager:
    """Production-grade serialization manager with hooks and validation"""
    
    def __init__(self):
        self.pre_serialize_hooks = []
        self.post_deserialize_hooks = []
        self.serialization_stats = {}
    
    def add_pre_serialize_hook(self, hook):
        """Add hook to run before serialization"""
        self.pre_serialize_hooks.append(hook)
    
    def add_post_deserialize_hook(self, hook):
        """Add hook to run after deserialization"""
        self.post_deserialize_hooks.append(hook)
    
    def serialize_with_hooks(self, obj, key=None):
        """Serialize object with pre/post hooks"""
        # Run pre-serialization hooks
        for hook in self.pre_serialize_hooks:
            obj = hook(obj, 'serialize')
        
        # Track serialization
        start_time = time.time()
        try:
            serialized = dill.dumps(obj)
            success = True
            error = None
        except Exception as e:
            serialized = None
            success = False
            error = str(e)
        
        elapsed = time.time() - start_time
        
        # Store statistics
        if key:
            self.serialization_stats[key] = {
                'size': len(serialized) if serialized else 0,
                'time': elapsed,
                'success': success,
                'error': error,
                'type': type(obj).__name__
            }
        
        return serialized
    
    def deserialize_with_hooks(self, data, key=None):
        """Deserialize object with post hooks"""
        start_time = time.time()
        try:
            obj = dill.loads(data)
            success = True
            error = None
            
            # Run post-deserialization hooks
            for hook in self.post_deserialize_hooks:
                obj = hook(obj, 'deserialize')
                
        except Exception as e:
            obj = None
            success = False
            error = str(e)
        
        elapsed = time.time() - start_time
        
        # Update statistics
        if key and key in self.serialization_stats:
            self.serialization_stats[key].update({
                'deserialize_time': elapsed,
                'deserialize_success': success,
                'deserialize_error': error
            })
        
        return obj

# Production decorators using dill
def memoize_with_dill(cache_file=None):
    """Memoization decorator that can persist function results with dill"""
    def decorator(func):
        cache = {}
        
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Create cache key
            key = dill.dumps((args, kwargs))
            
            if key not in cache:
                # Compute result
                result = func(*args, **kwargs)
                cache[key] = result
                
                # Optionally persist to file
                if cache_file:
                    try:
                        with open(cache_file, 'wb') as f:
                            dill.dump(cache, f)
                    except Exception:
                        pass  # Ignore file errors in production
            
            return cache[key]
        
        # Add cache management methods
        wrapper.clear_cache = lambda: cache.clear()
        wrapper.cache_info = lambda: {
            'size': len(cache),
            'keys': len(cache)
        }
        
        return wrapper
    return decorator

print("Production patterns and advanced features:")
print("=" * 43)

# Test 1: Advanced serialization manager
print("1. Advanced serialization with hooks:")

manager = SerializationManager()

# Add validation hooks
def validation_hook(obj, operation):
    """Validate objects during serialization"""
    if operation == 'serialize':
        if hasattr(obj, '__dict__') and len(obj.__dict__) > 100:
            print(f"    Warning: Large object with {len(obj.__dict__)} attributes")
    elif operation == 'deserialize':
        if hasattr(obj, 'validate') and callable(obj.validate):
            obj.validate()
    return obj

def logging_hook(obj, operation):
    """Log serialization operations"""
    print(f"    {operation.capitalize()}d: {type(obj).__name__}")
    return obj

manager.add_pre_serialize_hook(validation_hook)
manager.add_pre_serialize_hook(logging_hook)
manager.add_post_deserialize_hook(validation_hook)
manager.add_post_deserialize_hook(logging_hook)

# Test objects
class ValidatableData:
    def __init__(self, data):
        self.data = data
        self.processed = False
    
    def validate(self):
        if not isinstance(self.data, list):
            raise ValueError("Data must be a list")
        self.processed = True
        print(f"    Validation passed for {len(self.data)} items")

test_objects = {
    'simple_dict': {'a': 1, 'b': 2, 'c': 3},
    'complex_function': lambda x: x**2 + 2*x + 1,
    'validatable_object': ValidatableData([1, 2, 3, 4, 5])
}

for name, obj in test_objects.items():
    print(f"  Testing {name}:")
    
    # Serialize with hooks
    serialized = manager.serialize_with_hooks(obj, name)
    
    if serialized:
        # Deserialize with hooks
        restored = manager.deserialize_with_hooks(serialized, name)
        
        # Test functionality where applicable
        if name == 'complex_function' and restored:
            result = restored(3)  # Should be 9 + 6 + 1 = 16
            print(f"    Function result: f(3) = {result}")
            assert result == 16, result

# Show serialization statistics
print(f"\n  Serialization statistics:")
for key, stats in manager.serialization_stats.items():
    print(f"    {key}: {stats['type']}, {stats['size']} bytes, {stats['time']:.4f}s")

# the manager RECORDS a failed dumps/loads instead of raising, so the recorded outcome is what must be checked
assert sorted(manager.serialization_stats) == sorted(test_objects), manager.serialization_stats
assert all(s["success"] and s["deserialize_success"] for s in manager.serialization_stats.values()), manager.serialization_stats

# Test 2: Production memoization
print(f"\n2. Production memoization with dill:")

@memoize_with_dill()
def expensive_computation(n, operation='fibonacci'):
    """Expensive computation that benefits from memoization"""
    print(f"    Computing {operation} for n={n}")
    
    if operation == 'fibonacci':
        if n <= 1:
            return n
        return expensive_computation(n-1, operation) + expensive_computation(n-2, operation)
    elif operation == 'factorial':
        if n <= 1:
            return 1
        return n * expensive_computation(n-1, operation)
    else:
        return sum(i**2 for i in range(n))

# Test memoization
test_values = [10, 8, 10, 12, 8]  # Repeated values to show memoization

start_time = time.time()
for val in test_values:
    result = expensive_computation(val, 'fibonacci')
    
elapsed = time.time() - start_time
cache_info = expensive_computation.cache_info()

print(f"  Computed {len(test_values)} values in {elapsed:.4f}s")
print(f"  Cache info: {cache_info}")

# Test 3: Module and code serialization
print(f"\n3. Module and code object serialization:")

# Create dynamic code
code_string = """
def dynamic_function(x, y):
    '''Dynamically created function'''
    return x**2 + y**2 + 2*x*y

class DynamicClass:
    def __init__(self, value):
        self.value = value
    
    def compute(self):
        return self.value * 2 + 10

# Module-level variable
module_data = {'created_dynamically': True, 'version': 1.0}
"""

# Execute code to create objects
exec_globals = {}
exec(code_string, exec_globals)

# Extract created objects
dynamic_function = exec_globals['dynamic_function']
DynamicClass = exec_globals['DynamicClass']
module_data = exec_globals['module_data']

# Test serialization of dynamically created objects
dynamic_objects = {
    'dynamic_function': dynamic_function,
    'dynamic_class': DynamicClass,
    'dynamic_instance': DynamicClass(42),
    'module_data': module_data
}

print("  Serializing dynamically created objects:")
for name, obj in dynamic_objects.items():
    serialized = dill.dumps(obj)
    restored = dill.loads(serialized)
    
    # Test functionality
    if name == 'dynamic_function':
        result = restored(3, 4)  # Should be 9 + 16 + 24 = 49
        print(f"    {name}: Success, f(3,4) = {result}")
        assert result == 49, result
    elif name == 'dynamic_instance':
        result = restored.compute()  # Should be 42 * 2 + 10 = 94
        print(f"    {name}: Success, compute() = {result}")
        assert result == 94, result
    else:
        print(f"    {name}: Success ({len(serialized)} bytes)")
        

# Test 4: Compatibility and version handling
print(f"\n4. Dill compatibility and features:")

print(f"  Dill version: {dill.__version__}")
print(f"  Python version: {sys.version_info[:2]}")
print(f"  Pickle protocols supported: {list(range(dill.HIGHEST_PROTOCOL + 1))}")

# Test different pickle protocols
test_obj = {'data': [1, 2, 3], 'func': lambda x: x*2}

for protocol in [0, dill.HIGHEST_PROTOCOL]:
    serialized = dill.dumps(test_obj, protocol=protocol)
    restored = dill.loads(serialized)
    
    # Test restored function
    func_result = restored['func'](5)
    print(f"  Protocol {protocol}: {len(serialized)} bytes, func works: {func_result}")
    assert func_result == 10 and restored["data"] == [1, 2, 3], restored
    

print("\nProduction patterns and advanced features completed!")
print("Dill provides comprehensive serialization for production environments!")